# Charité Dataset - LOSO Cross-Validation with Sliding Windows
Data segmentation with Leave-One-Subject-Out (LOSO) for Freezing of Gait detection

## Setup and Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import LeaveOneGroupOut
import pickle
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

In [ ]:
# Configuration
SAMPLING_RATE = 200  # Hz
WINDOW_DURATION = 4.0  # seconds
WINDOW_SIZE = int(WINDOW_DURATION * SAMPLING_RATE)  # 800 samples
OVERLAP = 0.5  # 50%
STEP_SIZE = int(WINDOW_SIZE * (1 - OVERLAP))  # 400 samples
PRE_FOG_DURATION = 0.5  # seconds
PRE_FOG_SAMPLES = int(PRE_FOG_DURATION * SAMPLING_RATE)  # 100 samples

print(f"Window size: {WINDOW_SIZE} samples ({WINDOW_DURATION}s)")
print(f"Step size: {STEP_SIZE} samples ({STEP_SIZE/SAMPLING_RATE}s)")
print(f"Pre-FoG window: {PRE_FOG_SAMPLES} samples ({PRE_FOG_DURATION}s)")

## Load Preprocessed Dataset

In [ ]:
csv_path = Path(r'c:\Users\david\Documents\UADY_CARRERA\10_Decimo_Semestre\Seminario_II\Seminario_II\outputs\datasets_csv\charite_complete_dataset.csv')
data = pd.read_csv(csv_path)

print(f"Loaded {len(data):,} samples")
print(f"Columns: {list(data.columns)}")
print(f"\nLabel distribution:")
print(data['label'].value_counts())

In [ ]:
data.head()

## Create Binary Labels

In [ ]:
data['binary_label'] = (data['label'] == 1).astype(int)

print("Binary label distribution:")
print(data['binary_label'].value_counts())
print(f"\nClass 0 (No Freeze): {(data['binary_label'] == 0).sum():,} samples")
print(f"Class 1 (Freeze): {(data['binary_label'] == 1).sum():,} samples")

## Create Multiclass Labels with Pre-FoG Detection

In [ ]:
data['multiclass_label'] = data['binary_label'].copy()

# Mark Pre-FoG (0.5s before FoG onset)
for subject_id in data['subject_id'].unique():
    for trial_id in data[data['subject_id'] == subject_id]['trial_id'].unique():
        for foot in data[(data['subject_id'] == subject_id) & 
                         (data['trial_id'] == trial_id)]['foot'].unique():
            
            mask = (data['subject_id'] == subject_id) & \
                   (data['trial_id'] == trial_id) & \
                   (data['foot'] == foot)
            trial_data = data[mask].copy()
            
            binary_labels = trial_data['binary_label'].values
            multiclass_labels = binary_labels.copy()
            
            # Find FoG start transitions (0 -> 1)
            transitions = np.where((binary_labels[:-1] == 0) & (binary_labels[1:] == 1))[0]
            
            for trans_idx in transitions:
                start_idx = max(0, trans_idx - PRE_FOG_SAMPLES + 1)
                # Only mark as Pre-FoG if currently No Freeze
                for i in range(start_idx, trans_idx + 1):
                    if multiclass_labels[i] == 0:
                        multiclass_labels[i] = 2
            
            data.loc[mask, 'multiclass_label'] = multiclass_labels

print("Multiclass label distribution:")
print(data['multiclass_label'].value_counts())
print(f"\nClass 0 (No Freeze): {(data['multiclass_label'] == 0).sum():,} samples")
print(f"Class 1 (Freeze): {(data['multiclass_label'] == 1).sum():,} samples")
print(f"Class 2 (Pre-FoG): {(data['multiclass_label'] == 2).sum():,} samples")

## Visualize Label Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Binary
binary_counts = data['binary_label'].value_counts().sort_index()
axes[0].bar(['No Freeze', 'Freeze'], binary_counts.values, color=['#2ecc71', '#e74c3c'])
axes[0].set_ylabel('Number of Samples')
axes[0].set_title('Binary Label Distribution')
axes[0].grid(axis='y', alpha=0.3)

# Multiclass
multi_counts = data['multiclass_label'].value_counts().sort_index()
axes[1].bar(['No Freeze', 'Freeze', 'Pre-FoG'], multi_counts.values, 
            color=['#2ecc71', '#e74c3c', '#f39c12'])
axes[1].set_ylabel('Number of Samples')
axes[1].set_title('Multiclass Label Distribution')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Define Sliding Window Function

In [ ]:
def create_windows_per_subject(df_subset, feature_cols, label_col, window_size, step_size):
    """
    Create sliding windows maintaining temporal continuity per subject-trial-foot
    """
    windows = []
    labels = []
    subjects = []
    
    for subject_id in df_subset['subject_id'].unique():
        for trial_id in df_subset[df_subset['subject_id'] == subject_id]['trial_id'].unique():
            for foot in df_subset[(df_subset['subject_id'] == subject_id) & 
                                  (df_subset['trial_id'] == trial_id)]['foot'].unique():
                
                mask = (df_subset['subject_id'] == subject_id) & \
                       (df_subset['trial_id'] == trial_id) & \
                       (df_subset['foot'] == foot)
                segment = df_subset[mask].reset_index(drop=True)
                
                if len(segment) < window_size:
                    continue
                
                # Create sliding windows
                for start_idx in range(0, len(segment) - window_size + 1, step_size):
                    window = segment.iloc[start_idx:start_idx + window_size]
                    window_data = window[feature_cols].values
                    window_labels = window[label_col].values
                    
                    # Majority voting for label
                    window_label = Counter(window_labels).most_common(1)[0][0]
                    
                    windows.append(window_data)
                    labels.append(window_label)
                    subjects.append(subject_id)
    
    return np.array(windows), np.array(labels), np.array(subjects)

## LOSO Cross-Validation Setup

In [ ]:
# Feature columns
feature_cols = ['acc_x_m_s2', 'acc_y_m_s2', 'acc_z_m_s2', 
                'gyr_x_deg_s', 'gyr_y_deg_s', 'gyr_z_deg_s']

# Setup LOSO
logo = LeaveOneGroupOut()
subjects = data['subject_id'].values

print(f"Total subjects: {data['subject_id'].nunique()}")
print(f"Number of folds: {logo.get_n_splits(groups=subjects)}")

## Create LOSO Folds with Windows - Binary Classification

In [ ]:
binary_folds = []

for fold_idx, (train_idx, test_idx) in enumerate(logo.split(data, groups=subjects)):
    df_train = data.iloc[train_idx]
    df_test = data.iloc[test_idx]
    
    test_subject = df_test['subject_id'].iloc[0]
    
    # Create windows separately for train and test
    X_train, y_train, subjects_train = create_windows_per_subject(
        df_train, feature_cols, 'binary_label', WINDOW_SIZE, STEP_SIZE
    )
    
    X_test, y_test, subjects_test = create_windows_per_subject(
        df_test, feature_cols, 'binary_label', WINDOW_SIZE, STEP_SIZE
    )
    
    # Store fold information
    fold_dict = {
        'fold': fold_idx,
        'test_subject': int(test_subject),
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test,
        'subjects_train': subjects_train,
        'subjects_test': subjects_test,
        'train_dist': dict(Counter(y_train)),
        'test_dist': dict(Counter(y_test))
    }
    
    binary_folds.append(fold_dict)
    
    print(f"Fold {fold_idx}: Test subject {test_subject} | "
          f"Train: {len(X_train)} windows | Test: {len(X_test)} windows")

print(f"\nTotal folds: {len(binary_folds)}")

## Create LOSO Folds with Windows - Multiclass Classification

In [ ]:
multiclass_folds = []

for fold_idx, (train_idx, test_idx) in enumerate(logo.split(data, groups=subjects)):
    df_train = data.iloc[train_idx]
    df_test = data.iloc[test_idx]
    
    test_subject = df_test['subject_id'].iloc[0]
    
    # Create windows separately for train and test
    X_train, y_train, subjects_train = create_windows_per_subject(
        df_train, feature_cols, 'multiclass_label', WINDOW_SIZE, STEP_SIZE
    )
    
    X_test, y_test, subjects_test = create_windows_per_subject(
        df_test, feature_cols, 'multiclass_label', WINDOW_SIZE, STEP_SIZE
    )
    
    # Store fold information
    fold_dict = {
        'fold': fold_idx,
        'test_subject': int(test_subject),
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test,
        'subjects_train': subjects_train,
        'subjects_test': subjects_test,
        'train_dist': dict(Counter(y_train)),
        'test_dist': dict(Counter(y_test))
    }
    
    multiclass_folds.append(fold_dict)
    
    print(f"Fold {fold_idx}: Test subject {test_subject} | "
          f"Train: {len(X_train)} windows | Test: {len(X_test)} windows")

print(f"\nTotal folds: {len(multiclass_folds)}")

## Validation Checks

In [ ]:
# Check no subject overlap between train and test
for fold in binary_folds:
    train_subjects = set(fold['subjects_train'])
    test_subjects = set(fold['subjects_test'])
    overlap = train_subjects.intersection(test_subjects)
    assert len(overlap) == 0, f"Fold {fold['fold']}: Subject overlap detected!"

print("✓ No subject overlap between train/test sets")

# Check all subjects are covered
all_test_subjects = set([fold['test_subject'] for fold in binary_folds])
expected_subjects = set(data['subject_id'].unique())
assert all_test_subjects == expected_subjects, "Not all subjects covered!"

print(f"✓ All {len(all_test_subjects)} subjects covered in LOSO folds")

# Check window shapes
for fold in binary_folds:
    assert fold['X_train'].shape[1:] == (WINDOW_SIZE, len(feature_cols))
    assert fold['X_test'].shape[1:] == (WINDOW_SIZE, len(feature_cols))

print(f"✓ All windows have shape ({WINDOW_SIZE}, {len(feature_cols)})")

## Visualize Fold Distributions

In [ ]:
# Binary folds
fold_stats = []
for fold in binary_folds:
    fold_stats.append({
        'Fold': fold['fold'],
        'Test Subject': fold['test_subject'],
        'Train Windows': len(fold['X_train']),
        'Test Windows': len(fold['X_test'])
    })

fold_df = pd.DataFrame(fold_stats)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(fold_df['Fold'], fold_df['Train Windows'], color='#3498db', alpha=0.7)
axes[0].set_xlabel('Fold')
axes[0].set_ylabel('Number of Windows')
axes[0].set_title('Train Windows per Fold (Binary)')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(fold_df['Fold'], fold_df['Test Windows'], color='#e74c3c', alpha=0.7)
axes[1].set_xlabel('Fold')
axes[1].set_ylabel('Number of Windows')
axes[1].set_title('Test Windows per Fold (Binary)')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(fold_df)

In [ ]:
# Class distribution across folds
train_dist = []
test_dist = []

for fold in binary_folds:
    train_counts = fold['train_dist']
    test_counts = fold['test_dist']
    
    train_dist.append([train_counts.get(0, 0), train_counts.get(1, 0)])
    test_dist.append([test_counts.get(0, 0), test_counts.get(1, 0)])

train_dist = np.array(train_dist)
test_dist = np.array(test_dist)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(binary_folds))
width = 0.35

axes[0].bar(x - width/2, train_dist[:, 0], width, label='No Freeze', color='#2ecc71')
axes[0].bar(x + width/2, train_dist[:, 1], width, label='Freeze', color='#e74c3c')
axes[0].set_xlabel('Fold')
axes[0].set_ylabel('Number of Windows')
axes[0].set_title('Binary Train Class Distribution per Fold')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(x - width/2, test_dist[:, 0], width, label='No Freeze', color='#2ecc71')
axes[1].bar(x + width/2, test_dist[:, 1], width, label='Freeze', color='#e74c3c')
axes[1].set_xlabel('Fold')
axes[1].set_ylabel('Number of Windows')
axes[1].set_title('Binary Test Class Distribution per Fold')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Multiclass distribution
train_dist_multi = []
test_dist_multi = []

for fold in multiclass_folds:
    train_counts = fold['train_dist']
    test_counts = fold['test_dist']
    
    train_dist_multi.append([train_counts.get(0, 0), train_counts.get(1, 0), train_counts.get(2, 0)])
    test_dist_multi.append([test_counts.get(0, 0), test_counts.get(1, 0), test_counts.get(2, 0)])

train_dist_multi = np.array(train_dist_multi)
test_dist_multi = np.array(test_dist_multi)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(multiclass_folds))
width = 0.25

axes[0].bar(x - width, train_dist_multi[:, 0], width, label='No Freeze', color='#2ecc71')
axes[0].bar(x, train_dist_multi[:, 1], width, label='Freeze', color='#e74c3c')
axes[0].bar(x + width, train_dist_multi[:, 2], width, label='Pre-FoG', color='#f39c12')
axes[0].set_xlabel('Fold')
axes[0].set_ylabel('Number of Windows')
axes[0].set_title('Multiclass Train Distribution per Fold')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(x - width, test_dist_multi[:, 0], width, label='No Freeze', color='#2ecc71')
axes[1].bar(x, test_dist_multi[:, 1], width, label='Freeze', color='#e74c3c')
axes[1].bar(x + width, test_dist_multi[:, 2], width, label='Pre-FoG', color='#f39c12')
axes[1].set_xlabel('Fold')
axes[1].set_ylabel('Number of Windows')
axes[1].set_title('Multiclass Test Distribution per Fold')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Visualize Example Windows

In [ ]:
# Visualize one window from first fold
sample_fold = binary_folds[0]
sample_window = sample_fold['X_train'][0]
sample_label = sample_fold['y_train'][0]

fig, axes = plt.subplots(2, 1, figsize=(12, 6))

# Accelerometer
time_axis = np.arange(WINDOW_SIZE) / SAMPLING_RATE
axes[0].plot(time_axis, sample_window[:, 0], label='acc_x', alpha=0.7)
axes[0].plot(time_axis, sample_window[:, 1], label='acc_y', alpha=0.7)
axes[0].plot(time_axis, sample_window[:, 2], label='acc_z', alpha=0.7)
axes[0].set_ylabel('Acceleration (m/s²)')
axes[0].set_title(f'Sample Window - Label: {sample_label}')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Gyroscope
axes[1].plot(time_axis, sample_window[:, 3], label='gyr_x', alpha=0.7)
axes[1].plot(time_axis, sample_window[:, 4], label='gyr_y', alpha=0.7)
axes[1].plot(time_axis, sample_window[:, 5], label='gyr_z', alpha=0.7)
axes[1].set_xlabel('Time (seconds)')
axes[1].set_ylabel('Angular Velocity (deg/s)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Visualize Pre-FoG Detection Example

In [ ]:
# Find a subject-trial-foot with Pre-FoG
sample_data = None
for subject_id in data['subject_id'].unique():
    for trial_id in data[data['subject_id'] == subject_id]['trial_id'].unique():
        for foot in data[(data['subject_id'] == subject_id) & 
                         (data['trial_id'] == trial_id)]['foot'].unique():
            
            mask = (data['subject_id'] == subject_id) & \
                   (data['trial_id'] == trial_id) & \
                   (data['foot'] == foot)
            segment = data[mask]
            
            if (segment['multiclass_label'] == 2).sum() > 0:
                sample_data = segment
                break
        if sample_data is not None:
            break
    if sample_data is not None:
        break

if sample_data is not None:
    # Find first Pre-FoG occurrence
    pre_fog_idx = sample_data[sample_data['multiclass_label'] == 2].index[0]
    start_idx = max(pre_fog_idx - 400, sample_data.index[0])
    end_idx = min(pre_fog_idx + 400, sample_data.index[-1])
    
    plot_segment = sample_data.loc[start_idx:end_idx].reset_index(drop=True)
    
    fig, ax = plt.subplots(figsize=(14, 5))
    
    time_axis = np.arange(len(plot_segment)) / SAMPLING_RATE
    
    # Plot accelerometer magnitude
    acc_mag = np.sqrt(plot_segment['acc_x_m_s2']**2 + 
                      plot_segment['acc_y_m_s2']**2 + 
                      plot_segment['acc_z_m_s2']**2)
    ax.plot(time_axis, acc_mag, color='black', linewidth=1.5)
    
    # Color background by label
    for label, color in [(0, '#2ecc71'), (2, '#f39c12'), (1, '#e74c3c')]:
        label_mask = plot_segment['multiclass_label'] == label
        if label_mask.any():
            ax.fill_between(time_axis, 0, acc_mag.max() * 1.1, 
                           where=label_mask, alpha=0.3, color=color,
                           label=['No Freeze', 'Freeze', 'Pre-FoG'][label])
    
    ax.set_xlabel('Time (seconds)')
    ax.set_ylabel('Acceleration Magnitude (m/s²)')
    ax.set_title('Pre-FoG Detection Example')
    ax.legend()
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("No Pre-FoG examples found in dataset")

## Save to Pickle Files

In [ ]:
output_dir = Path(r'c:\Users\david\Documents\UADY_CARRERA\10_Decimo_Semestre\Seminario_II\Seminario_II\examples\charite_dataset\datasets')
output_dir.mkdir(parents=True, exist_ok=True)

# Save binary folds
binary_path = output_dir / 'charite_loso_windows_binary.pkl'
with open(binary_path, 'wb') as f:
    pickle.dump(binary_folds, f)
print(f"Binary folds saved to: {binary_path}")

# Save multiclass folds
multiclass_path = output_dir / 'charite_loso_windows_multiclass.pkl'
with open(multiclass_path, 'wb') as f:
    pickle.dump(multiclass_folds, f)
print(f"Multiclass folds saved to: {multiclass_path}")

## Summary

In [ ]:
print("=" * 60)
print("CHARITÉ DATASET - LOSO CROSS-VALIDATION SUMMARY")
print("=" * 60)
print(f"\nDataset Configuration:")
print(f"  Sampling rate: {SAMPLING_RATE} Hz")
print(f"  Window size: {WINDOW_SIZE} samples ({WINDOW_DURATION}s)")
print(f"  Overlap: {OVERLAP*100}%")
print(f"  Step size: {STEP_SIZE} samples")
print(f"  Pre-FoG window: {PRE_FOG_SAMPLES} samples ({PRE_FOG_DURATION}s)")

print(f"\nBinary Classification:")
print(f"  Total folds: {len(binary_folds)}")
print(f"  Classes: 2 (No Freeze, Freeze)")
total_train_windows = sum([len(f['X_train']) for f in binary_folds])
total_test_windows = sum([len(f['X_test']) for f in binary_folds])
print(f"  Total train windows: {total_train_windows:,}")
print(f"  Total test windows: {total_test_windows:,}")

print(f"\nMulticlass Classification:")
print(f"  Total folds: {len(multiclass_folds)}")
print(f"  Classes: 3 (No Freeze, Freeze, Pre-FoG)")
total_train_windows = sum([len(f['X_train']) for f in multiclass_folds])
total_test_windows = sum([len(f['X_test']) for f in multiclass_folds])
print(f"  Total train windows: {total_train_windows:,}")
print(f"  Total test windows: {total_test_windows:,}")

print(f"\nOutput Files:")
print(f"  Binary: {binary_path}")
print(f"  Multiclass: {multiclass_path}")
print("\n" + "=" * 60)